In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "easyocr", "spacy", "pandas", "openpyxl", "pillow", "tqdm"])
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

0

In [4]:
import os, re, json
import easyocr
import spacy
import pandas as pd
from PIL import Image
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────
DATA_DIR   = "data/raw"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load models ────────────────────────────────────────────────
print("Loading EasyOCR...")
reader = easyocr.Reader(["en"], gpu=False)  # change to True if you have GPU
print("Loading spaCy...")
nlp = spacy.load("en_core_web_sm")
print("Models ready!\n")

# ══════════════════════════════════════════════════════════════
#  SECTION 1 — PREPROCESSING
# ══════════════════════════════════════════════════════════════

def preprocess_image(img_path):
    """Convert to RGB, resize if too small"""
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    if w < 800:
        img = img.resize((800, int(h * 800 / w)), Image.LANCZOS)
    return img

# ══════════════════════════════════════════════════════════════
#  SECTION 2 — OCR
# ══════════════════════════════════════════════════════════════

def run_ocr(img_path):
    """Run EasyOCR and return list of text lines"""
    img = preprocess_image(img_path)
    # Save preprocessed image temporarily
    temp_path = "temp_preprocessed.png"
    img.save(temp_path)
    results = reader.readtext(temp_path, detail=0, paragraph=True)
    os.remove(temp_path)
    return results

# ══════════════════════════════════════════════════════════════
#  SECTION 3 — FIELD EXTRACTION
# ══════════════════════════════════════════════════════════════

# ── Email ──
def extract_email(text):
    match = re.findall(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", text)
    return match[0] if match else ""

# ── Phone ──
def extract_phone(text):
    match = re.findall(r"[\+\(]?[1-9][0-9\s\-\(\)]{7,}[0-9]", text)
    return match[0].strip() if match else ""

# ── Name — first non-empty line or spaCy PERSON entity ──
def extract_name(lines, doc):
    # Try spaCy first
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            return ent.text
    # Fallback: first line
    for line in lines:
        if len(line.strip()) > 2:
            return line.strip()
    return ""

# ── Location ──
def extract_location(doc):
    for ent in doc.ents:
        if ent.label_ in ["GPE", "LOC"]:
            return ent.text
    return ""

# ── Section detector ──
SECTION_KEYWORDS = {
    "skills":     ["skill", "skills", "technical skills", "core competencies",
                   "technologies", "tools", "expertise"],
    "education":  ["education", "academic", "qualification", "degree",
                   "university", "college", "school"],
    "experience": ["experience", "work experience", "employment", "career",
                   "professional experience", "internship", "projects"],
}

def detect_sections(lines):
    """
    Returns dict with section name → list of lines belonging to that section
    """
    sections = {k: [] for k in SECTION_KEYWORDS}
    current_section = None

    for line in lines:
        line_lower = line.lower().strip()
        matched = False
        for section, keywords in SECTION_KEYWORDS.items():
            if any(kw in line_lower for kw in keywords):
                current_section = section
                matched = True
                break
        if not matched and current_section:
            sections[current_section].append(line.strip())

    return sections

def clean_section(lines):
    """Join lines, remove duplicates, strip blanks"""
    seen = set()
    clean = []
    for l in lines:
        l = l.strip()
        if l and l not in seen:
            seen.add(l)
            clean.append(l)
    return " | ".join(clean)

# ══════════════════════════════════════════════════════════════
#  SECTION 4 — MAIN PIPELINE
# ══════════════════════════════════════════════════════════════

SUPPORTED = (".jpg", ".jpeg", ".png", ".webp")
image_files = [f for f in os.listdir(DATA_DIR)
               if f.lower().endswith(SUPPORTED)]

print(f"Found {len(image_files)} resume images\n")

records = []

for fname in tqdm(image_files):
    img_path = os.path.join(DATA_DIR, fname)

    try:
        # OCR
        lines = run_ocr(img_path)
        full_text = " ".join(lines)

        # spaCy
        doc = nlp(full_text[:10000])  # limit to avoid memory issues

        # Extract fields
        name     = extract_name(lines, doc)
        email    = extract_email(full_text)
        phone    = extract_phone(full_text)
        location = extract_location(doc)

        # Section detection
        sections   = detect_sections(lines)
        skills     = clean_section(sections["skills"])
        education  = clean_section(sections["education"])
        experience = clean_section(sections["experience"])

        records.append({
            "file":       fname,
            "name":       name,
            "email":      email,
            "phone":      phone,
            "location":   location,
            "skills":     skills,
            "education":  education,
            "experience": experience,
        })

    except Exception as e:
        print(f"Error on {fname}: {e}")
        records.append({"file": fname, "name": "", "email": "",
                        "phone": "", "location": "", "skills": "",
                        "education": "", "experience": ""})

# ══════════════════════════════════════════════════════════════
#  SECTION 5 — EXPORT TO EXCEL
# ══════════════════════════════════════════════════════════════

df = pd.DataFrame(records)
output_path = os.path.join(OUTPUT_DIR, "resume_output.xlsx")
df.to_excel(output_path, index=False)
print(f"\nDone! Saved to {output_path}")
print(f"Total resumes processed: {len(df)}")

# ══════════════════════════════════════════════════════════════
#  SECTION 6 — QUICK METRICS
# ══════════════════════════════════════════════════════════════

print("\n" + "="*50)
print("  EXTRACTION METRICS")
print("="*50)
fields = ["name", "email", "phone", "location", "skills", "education", "experience"]
for f in fields:
    filled = df[f].apply(lambda x: 1 if str(x).strip() else 0).sum()
    pct    = filled / len(df) * 100
    print(f"  {f:<12} extracted in {filled}/{len(df)} resumes → {pct:.1f}%")

print("\nOpen output/resume_output.xlsx to see your structured data!")

Using CPU. Note: This module is much faster with a GPU.


Loading EasyOCR...
Loading spaCy...
Models ready!

Found 50 resume images



  0%|          | 0/50 [00:00<?, ?it/s]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  2%|▏         | 1/50 [00:29<24:16, 29.73s/it]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  4%|▍         | 2/50 [00:45<17:25, 21.78s/it]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  6%|▌         | 3/50 [01:06<16:30, 21.08s/it]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerato


Done! Saved to output\resume_output.xlsx
Total resumes processed: 50

  EXTRACTION METRICS
  name         extracted in 50/50 resumes → 100.0%
  email        extracted in 19/50 resumes → 38.0%
  phone        extracted in 46/50 resumes → 92.0%
  location     extracted in 41/50 resumes → 82.0%
  skills       extracted in 40/50 resumes → 80.0%
  education    extracted in 36/50 resumes → 72.0%
  experience   extracted in 38/50 resumes → 76.0%

Open output/resume_output.xlsx to see your structured data!
